# Extract CONCH (VLM) image + text features

Run once per (dataset, seed, VLM, description style). Unlike DINOv2, a VLM has
**two image embedding spaces** that are not interchangeable
(`features/vlm.py` module docstring has the full detail):

* `RAW_SPACE` (`proj_contrast=False, normalize=False`) -- for the linear probe,
  the coverage kernel, everything `scalpel`'s disagreement machinery reads.
* `PROJ_SPACE` (`proj_contrast=True, normalize=True`) -- for comparing an
  image against text, which is what the round-1 cold-start prior needs.

Both are written in the **same** forward pass, because 448x448 (CONCH's
resolution, 4x the pixels of DINOv2's 224x224) makes re-running the whole
dataloader for the other space a real cost, not a convenience.

This also builds the **text prototypes**: one 512-d vector per class, either
from the official CONCH CRC100K prompt set (`conch_official` -- vendored in
`config/prompts/`, PathMNIST only) or from a description file
`generate_class_description.ipynb` wrote. `manual` reads
`datasets.<name>.descriptions` from `config.yaml` directly, no file needed.

**One configuration per run** (`DATASET`, `SEED`, `VLM`, `DESCRIPTION_STYLE` are
all singular): the notebook ends in one zip whose name states the whole
configuration. Sweeping a style means running the notebook again.

**Two Kaggle Datasets required**, same reasoning as `run_al_baseline.ipynb`:
the raw images (`DATA_ROOT`) and nothing else -- there is no VLM cache to
attach yet on a first run, since this notebook is what produces one.

Zero-shot accuracy on the pool, using whichever text prototype this run built,
is printed and asserted `> 0.70` at the end -- not to reproduce the paper's
79.1% (PathMNIST is 224-native, resized up to CONCH's 448, so an exact match
is not expected), but to catch a wrong transform, normalization, tokenizer or
projection before it silently corrupts every cold-start result built on this
cache. A near-11%-random score means something upstream is wrong.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import os
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf-transfer"])
# open_clip_torch is NOT enough: CONCH has its own tokenizer + factory
# (features/vlm.py module docstring, section on the tokenizer). Install the
# conch package itself.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "git+https://github.com/mahmoodlab/CONCH.git"])

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")

In [ ]:
# ---- EDIT THIS CELL ----
DATASET = "pathmnist"
SEED = 42                          # ONE seed, ONE zip

# CONCH | Astaxanthin/KEEP | wisdomik/QuiltNet-B-16-PMB | vinid/plip
# Only CONCH is verified against the paper/code in PLAN_IMPLEMENT.md §10 --
# a different VLM likely needs its own loader in features/vlm.py.
VLM = "MahmoodLab/CONCH"

# manual | conch_official | llm_short | llm_morphology | llm_multi
#   manual         : datasets.<DATASET>.descriptions in config.yaml, no file
#   conch_official : the paper authors' own 22-template x 4-5-classname
#                     ensemble for CRC100K -- PathMNIST ONLY, vendored in
#                     config/prompts/. The baseline any LLM description must
#                     beat to justify itself as a contribution.
#   llm_*          : written by generate_class_description.ipynb; must have
#                     been run first, or this notebook fails asking for it.
DESCRIPTION_STYLE = "conch_official"

HF_TOKEN = ""                      # CONCH is gated on HF -- needed to download it

# Two Kaggle Datasets, same split as run_al_baseline.ipynb: the raw images
# (labels + fingerprint) are required even though this notebook is what
# BUILDS the VLM cache -- there is nothing to attach for that side yet.
DATA_ROOT = "/kaggle/input/datasets/cryandrrich/nckh2026"
FEATURE_DIR = "/kaggle/working/vlm_features"

# Use both T4s. 448x448 is 4x DINOv2's pixel count, so batch size is smaller
# than the DINOv2 extraction notebook uses.
PARALLEL = True
BATCH_SIZE = 64

In [ ]:
from huggingface_hub import login

if HF_TOKEN:
    login(HF_TOKEN)
else:
    print("[auth] HF_TOKEN is empty -- relying on a Kaggle Secret or a prior")
    print("       `huggingface-cli login` in this session. CONCH is gated: if")
    print("       neither is set up, the download cell below will fail with a")
    print("       401/403 from the Hub, not a silent wrong result.")

In [ ]:
import json

import numpy as np
import yaml
import torch

from data.loaders import get_data_loaders, get_sample_ids
from data.identity import sample_order_fingerprint
from features.vlm import (
    RAW_SPACE,
    PROJ_SPACE,
    assert_class_order_matches_prompts,
    description_sha256,
    encode_text_prototypes,
    get_or_extract_vlm_features,
    load_conch,
    load_official_conch_prompts,
    text_prototype_cache_paths,
    vlm_feature_cache_paths,
    zero_shot_logits,
)
from utils import vlm_archive_stem
from utils.kaggle import find_data_root
from utils.parallel import visible_gpu_count

In [ ]:
DATA_ROOT_RESOLVED = find_data_root([Path(DATA_ROOT)])

DATA_PATHS = {
    "pathmnist": str(DATA_ROOT_RESOLVED / "pathmnist_224.npz"),
    "histoset": str(DATA_ROOT_RESOLVED / "HistoSet-5x14/HistoSet-5x14"),
    "skintissue": str(DATA_ROOT_RESOLVED / "SkinTissue/SkinTissue/tiles"),
}
print("data root (raw images):", DATA_ROOT_RESOLVED)

In [ ]:
with open("config/config.yaml", "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

assert isinstance(SEED, int), "SEED is one seed, not a list -- re-run the notebook to sweep"
data_path = Path(DATA_PATHS[DATASET])
assert data_path.exists(), f"Missing Kaggle input: {data_path}"
assert torch.cuda.is_available(), "Attach a Kaggle GPU before extraction"
assert not str(FEATURE_DIR).startswith("/kaggle/input"), (
    "FEATURE_DIR must be writable; /kaggle/input is read-only"
)
Path(FEATURE_DIR).mkdir(parents=True, exist_ok=True)

# class_names in config.yaml order -- the order the probe/coverage side of the
# pipeline already uses everywhere else, so the text prototypes below are
# built in that same order and nothing needs a translation table.
dataset_info = config["datasets"][DATASET]
class_names = list(dataset_info["descriptions"])
num_classes = dataset_info["num_classes"]
assert len(class_names) == num_classes, (
    f"config.yaml lists {len(class_names)} descriptions but "
    f"num_classes={num_classes} for {DATASET!r}"
)

# Resolve DESCRIPTION_STYLE into a (class_names -> list-of-prompt-strings) map,
# built once here so the extraction cell below is style-agnostic.
if DESCRIPTION_STYLE == "manual":
    class_prompts = [[dataset_info["descriptions"][name]] for name in class_names]
    description_source = "config.yaml datasets.<dataset>.descriptions"
    description_hash = description_sha256(dataset_info["descriptions"])

elif DESCRIPTION_STYLE == "conch_official":
    assert DATASET == "pathmnist", (
        "the official CONCH prompt set is CRC100K-specific (9 classes) and "
        f"only matches PathMNIST's classes, not {DATASET!r}"
    )
    prompts = load_official_conch_prompts("config/prompts/crc100k_prompts_all_per_class.json")
    assert_class_order_matches_prompts(class_names, prompts["classnames"])
    # Official ensemble: every (classname x template) pair, per class -- see
    # features/vlm.py::encode_text_prototypes for why this must not be
    # collapsed to one string per class before encoding.
    templates = prompts["templates"]
    class_prompts = [
        [template.replace("CLASSNAME", classname)
         for classname in prompts["classnames"][code]
         for template in templates]
        for code in prompts["classnames"]
    ]
    description_source = "config/prompts/crc100k_prompts_all_per_class.json (official CONCH prompts)"
    description_hash = description_sha256(prompts["classnames"])

else:
    # llm_short | llm_morphology | llm_multi -- written by
    # generate_class_description.ipynb into config/descriptions/.
    description_path = Path(f"config/descriptions/{DATASET}_{DESCRIPTION_STYLE}.json")
    assert description_path.is_file(), (
        f"No description file at {description_path}. Run "
        "generate_class_description.ipynb with this DATASET/STYLE first."
    )
    with open(description_path, "r", encoding="utf-8") as handle:
        description_payload = json.load(handle)
    assert list(description_payload["descriptions"]) == class_names, (
        f"{description_path} class order does not match config.yaml's -- "
        "regenerate it against the current config"
    )
    class_prompts = [[description_payload["descriptions"][name]] for name in class_names]
    description_source = str(description_path)
    description_hash = description_payload.get("sha256") or description_sha256(
        description_payload["descriptions"]
    )

print(f"VLM: {VLM} | description_style: {DESCRIPTION_STYLE}")
print(f"description source: {description_source}")
print(f"classes ({len(class_names)}): {class_names}")
print(f"GPUs visible: {visible_gpu_count()}")

In [ ]:
import time

# No sharding across GPUs here, unlike the DINOv2 extraction notebook: this
# runs on ONE GPU (`torch.device("cuda:0")`). 448x448 already keeps a session
# well within a T4x2's 12-hour limit for a single dataset at this project's
# scale (see PLAN_IMPLEMENT.md 11.1 for the not-yet-benchmarked caveat) --
# sharding this too would be premature complexity for a notebook that already
# has plenty. PARALLEL is reserved for a future two-GPU split if extraction
# time turns out to need it.
started = time.time()
device = torch.device("cuda:0")
print(f"{DATASET} | seed {SEED} | {VLM}")

# The model must be loaded BEFORE the loaders, because building a loader with
# the right pixels requires CONCH's own `preprocess` (448x448 + OpenAI CLIP
# normalization) -- `get_data_loaders`'s default is DINOv2's 224+ImageNet,
# wrong for this model. Loaded once here and passed into
# `get_or_extract_vlm_features` below, so a cache MISS does not load the
# checkpoint a second time.
conch_model, conch_preprocess = load_conch(VLM, device, hf_token=HF_TOKEN)

train_loader, test_loader, _ = get_data_loaders(
    DATA_PATHS[DATASET], SEED, verbose=True, transform=conch_preprocess,
)
train_fingerprint = sample_order_fingerprint(get_sample_ids(train_loader.dataset))
test_fingerprint = sample_order_fingerprint(get_sample_ids(test_loader.dataset))
# `get_data_loaders`'s own batch size (256) assumes DINOv2's 224x224; CONCH's
# 448x448 is 4x the pixels, so rebuild the loaders at BATCH_SIZE instead.
train_loader = torch.utils.data.DataLoader(
    train_loader.dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=train_loader.num_workers, pin_memory=True,
)
test_loader = torch.utils.data.DataLoader(
    test_loader.dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=test_loader.num_workers, pin_memory=True,
)

cached = get_or_extract_vlm_features(
    train_loader, test_loader, DATASET, SEED, VLM, device,
    cache_dir=FEATURE_DIR,
    train_fingerprint=train_fingerprint, test_fingerprint=test_fingerprint,
    model=conch_model, hf_token=HF_TOKEN,
)
print(f"raw:  train {cached['train'].shape} | test {cached['test'].shape}")
print(f"proj: train {cached['proj_train'].shape} | test {cached['proj_test'].shape}")
print(f"total {time.time() - started:.0f}s")

In [ ]:
# Text prototypes: one 512-d vector per class, in PROJ_SPACE (the space image
# embeddings are compared against). Cached separately from the image features
# -- see features/vlm.py::text_prototype_cache_paths -- because it depends on
# (dataset, style) only, not on seed or the train/test split.
text_paths = text_prototype_cache_paths(FEATURE_DIR, DATASET, DESCRIPTION_STYLE)

if Path(text_paths["prototypes"]).is_file() and Path(text_paths["manifest"]).is_file():
    with open(text_paths["manifest"], "r", encoding="utf-8") as handle:
        text_manifest = json.load(handle)
    prototypes_match = (
        text_manifest.get("class_names") == class_names
        and text_manifest.get("description_sha256") == description_hash
    )
    if prototypes_match:
        text_prototypes = np.load(text_paths["prototypes"])
        print(f"[text] Loaded cache -> {text_paths['prototypes']} {text_prototypes.shape}")
    else:
        print("[text] Cache exists but class order or description changed -- recomputing.")
else:
    prototypes_match = False

if not prototypes_match:
    text_prototypes_t = encode_text_prototypes(conch_model, class_names, class_prompts, device)
    text_prototypes = text_prototypes_t.cpu().numpy().astype(np.float32)
    os.makedirs(FEATURE_DIR, exist_ok=True)
    np.save(text_paths["prototypes"], text_prototypes)
    with open(text_paths["manifest"], "w", encoding="utf-8") as handle:
        json.dump({
            "dataset": DATASET,
            "style": DESCRIPTION_STYLE,
            "vlm": VLM,
            "class_names": class_names,
            "description_source": description_source,
            "description_sha256": description_hash,
            "prompts_per_class": [len(p) for p in class_prompts],
        }, handle, indent=2, sort_keys=True)
    print(f"[text] Saved -> {text_paths['prototypes']} {text_prototypes.shape}")

# LEARNED, not the log(1/0.07) init value -- training moves it, and the
# official zero-shot code reads it off the loaded checkpoint every time
# (features/vlm.py module docstring, logit_scale section). Captured here,
# before the model is freed, since nothing after this needs the model itself.
conch_logit_scale = conch_model.logit_scale.exp().item()

del conch_model
if device.type == "cuda":
    torch.cuda.empty_cache()

In [ ]:
# Zero-shot check: not a reproduction of the paper's 79.1% (PathMNIST is
# 224-native, resized up to CONCH's 448 here, so an exact match is not
# expected -- see PLAN_IMPLEMENT.md 4.2), but a check that transform,
# normalization, tokenizer and projection are all correct. A wrong one of
# those does not crash; it silently produces near-random accuracy (~11% for
# 9 classes) with no other symptom.
import main as _main  # for the same label-extraction helper main.run() uses

test_labels = _main._dataset_labels(test_loader.dataset)
probs = zero_shot_logits(cached["proj_test"], text_prototypes, logit_scale=conch_logit_scale)
predictions = probs.argmax(axis=1)
zero_shot_accuracy = float((predictions == np.asarray(test_labels)).mean())

print(f"zero-shot accuracy on {DATASET} test ({len(test_labels)} images): "
      f"{zero_shot_accuracy:.4f}")
assert zero_shot_accuracy > 0.70, (
    f"zero-shot accuracy {zero_shot_accuracy:.4f} is far below the ~0.79 the "
    "paper reports on CRC100K -- check transform/normalization/tokenizer/"
    "projection before trusting this cache for anything downstream. A score "
    f"near 1/{num_classes} = {1 / num_classes:.3f} means something upstream "
    "is silently wrong, not that this dataset is simply harder."
)

In [ ]:
# Verify what will be shipped: both spaces present, complete, finite, and
# consistent with the manifests. Cheaper to fail here than after the zip has
# been published and attached to a run notebook.
vlm_paths = vlm_feature_cache_paths(FEATURE_DIR, DATASET, SEED, VLM)
for split in ("train", "test", "proj_train", "proj_test"):
    array = np.load(vlm_paths[split], mmap_mode="r")
    assert np.all(np.isfinite(array[:256])), f"{split} features are not all finite"
with open(vlm_paths["manifest"], "r", encoding="utf-8") as handle:
    manifest = json.load(handle)
with open(vlm_paths["proj_manifest"], "r", encoding="utf-8") as handle:
    proj_manifest = json.load(handle)
assert manifest["space"] == RAW_SPACE and proj_manifest["space"] == PROJ_SPACE
assert manifest["dataset"] == DATASET and manifest["seed"] == SEED
assert manifest["backbone"] == VLM

assert Path(text_paths["prototypes"]).is_file()
assert text_prototypes.shape == (num_classes, text_prototypes.shape[1])
print(f"OK {DATASET} seed{SEED} {VLM}: image {cached['train'].shape}/{cached['proj_train'].shape}, "
      f"text {text_prototypes.shape}, zero-shot acc {zero_shot_accuracy:.4f}")

In [ ]:
# Package the cache as ONE zip at the top of /kaggle/working, then delete the
# loose files -- the same shape every other publishing notebook in this
# project uses.
#
# Kaggle's Output tab lists what is left in /kaggle/working when the session
# ends, and in a "Save & Run All" session that is the ONLY way to get a file
# out: there is no terminal and no kaggle CLI. Keeping the originals beside
# the zip also doubles the download, and a session over the ~20 GB Output
# quota shows NOTHING at all, including the files that were fine.
import shutil

SOURCE = Path(FEATURE_DIR)
WORKING = Path("/kaggle/working")
assert SOURCE.is_dir(), f"nothing to archive at {SOURCE}"
assert SOURCE.resolve() != WORKING.resolve(), (
    "FEATURE_DIR must be a subdirectory of /kaggle/working, not /kaggle/working itself"
)

STEM = vlm_archive_stem(DATASET, SEED, VLM, DESCRIPTION_STYLE)
ARCHIVE = WORKING / STEM
shutil.make_archive(str(ARCHIVE), "zip", root_dir=SOURCE)
size_mb = ARCHIVE.with_suffix(".zip").stat().st_size / 1e6

print(f"{ARCHIVE.name}.zip  ({size_mb:.1f} MB) contains:")
for path in sorted(SOURCE.iterdir()):
    print(f"    {path.name}  ({path.stat().st_size / 1e6:.2f} MB)")

shutil.rmtree(SOURCE, ignore_errors=True)

remaining = sorted(p for p in WORKING.iterdir() if p.name != "codapath")
total_mb = sum(
    f.stat().st_size for p in remaining for f in ([p] if p.is_file() else p.rglob("*"))
    if f.is_file()
) / 1e6
print(f"\n/kaggle/working now holds {total_mb:.1f} MB (Output quota ~20 GB):")
for path in remaining:
    print(f"    {path.name}{'/' if path.is_dir() else ''}")

print(f"""
NEXT STEPS (no terminal needed)
  1. Output tab (right panel) -> download {ARCHIVE.name}.zip
  2. kaggle.com/datasets -> New Dataset -> upload that zip
     Kaggle extracts it into a directory named after the zip, so the cache
     files end up one level down. That is expected.
  3. In run_al_main.ipynb: Add Data -> your new dataset. The VLM feature
     cache is resolved by filename, so there is no path to edit.""")